# 03 Classification and QA

This notebook loads the December first-pass events, computes the Shinoda-style monsoon and vorticity metrics, and saves classification plus sensitivity summaries.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = "/content/JPCZcatalog"
if not os.path.exists(REPO_DIR):
    raise FileNotFoundError("Run notebooks/01_data_access_and_masks.ipynb first so the repo is cloned into /content/JPCZcatalog.")

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

events_path = Path("outputs/verification/december_events_first_pass.csv")
d_series_path = Path("outputs/verification/december_D_timeseries.nc")
if not events_path.exists():
    raise FileNotFoundError("Run notebooks/02_december_benchmark_detector.ipynb first to create outputs/verification/december_events_first_pass.csv.")
if not d_series_path.exists():
    raise FileNotFoundError("Run notebooks/02_december_benchmark_detector.ipynb first to create outputs/verification/december_D_timeseries.nc.")

print("Working directory:", os.getcwd())
print("Events file:", events_path)
print("D series file:", d_series_path)


In [ ]:
import pandas as pd

from jpcz_catalog.classify import (
    annotate_events_with_environment,
    assign_shinoda_classes,
    cache_type1_mean_winds,
    compute_december_climatological_slp_index,
    default_vorticity_box_variants,
    evaluate_threshold_box_combinations,
    evaluate_vorticity_box_sensitivity,
)
from jpcz_catalog.config import VORTICITY_BOX
from jpcz_catalog.detect import threshold_from_std
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.verification import (
    render_classification_summary,
    render_combo_sensitivity_summary,
    render_next_steps_summary,
    write_text_summary,
)

events_df = pd.read_csv(
    events_path,
    parse_dates=["event_start", "event_end", "event_peak"],
)
ds = open_arco_era5()
classified_events = annotate_events_with_environment(ds, events_df)
all_dec_slp_index, slp_clim_mean, slp_clim_std = compute_december_climatological_slp_index(ds)
classified_events, class_meta = assign_shinoda_classes(
    classified_events,
    slp_threshold=slp_clim_mean,
)

classified_events.to_csv("outputs/verification/december_events_shinoda_style.csv", index=False)
classification_summary = render_classification_summary(
    classified_events,
    slp_clim_mean=slp_clim_mean,
    slp_clim_std=slp_clim_std,
)
write_text_summary("outputs/verification/december_shinoda_style_summary.md", classification_summary)

print(classification_summary)
classified_events.head()


In [ ]:
import xarray as xr

all_dec_D = xr.open_dataarray(d_series_path).load()

uv12h_cache = cache_type1_mean_winds(ds, classified_events)
box_variants = default_vorticity_box_variants(VORTICITY_BOX)
box_sensitivity_df, type1_box_zeta = evaluate_vorticity_box_sensitivity(
    classified_events,
    uv12h_cache,
    box_variants=box_variants,
)
box_sensitivity_df.to_csv("outputs/verification/vorticity_box_sensitivity.csv", index=False)

D_mean, D_std, _ = threshold_from_std(all_dec_D, n_std=2.0)
combo_df = evaluate_threshold_box_combinations(
    classified_events,
    all_dec_D,
    D_mean=D_mean,
    D_std=D_std,
    slp_threshold=slp_clim_mean,
    box_variants={
        "original": box_variants["original"],
        "west_1deg": box_variants["west_1deg"],
        "south_1deg": box_variants["south_1deg"],
    },
    type1_box_zeta=type1_box_zeta,
    n_std_values=(2.0, 2.2, 2.3),
)
combo_df.to_csv("outputs/verification/combo_sensitivity_table.csv", index=False)
write_text_summary("outputs/verification/combo_sensitivity_summary.md", render_combo_sensitivity_summary())
write_text_summary("outputs/verification/next_steps.md", render_next_steps_summary())

print(box_sensitivity_df)
combo_df
